# Spam Guard Training

In [1]:
from sklearnex import patch_sklearn
patch_sklearn()

import joblib
import pandas as pd
from pathlib import Path
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import VotingClassifier
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, classification_report, confusion_matrix, f1_score, log_loss, precision_score, recall_score)
from sklearn.model_selection import GridSearchCV
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import RidgeClassifier
from sklearn.svm import SVC
from tqdm import tqdm


Extension for Scikit-learn* enabled (https://github.com/uxlfoundation/scikit-learn-intelex)


In [2]:
print("Loading vectors...")
x_tr = joblib.load('../vectors/x_training_vector.pkl')
x_te = joblib.load('../vectors/x_testing_vector.pkl')
y_tr = joblib.load('../vectors/y_training_vector.pkl')
y_te = joblib.load('../vectors/y_testing_vector.pkl')

print(f'x_tr shape: {x_tr.shape}')
print(f'x_te shape: {x_te.shape}')
print(f'y_tr length: {len(y_tr)}')
print(f'y_te length: {len(y_te)}')

Loading vectors...
x_tr shape: (326222, 5000)
x_te shape: (102111, 5000)
y_tr length: 326222
y_te length: 102111


## Feature Selection
Use chi-square to keep top 1000 features.

In [3]:
print("Running feature selection...")
sel = SelectKBest(chi2, k=1000)
x_tr_s = sel.fit_transform(x_tr, y_tr)
x_te_s = sel.transform(x_te)

print(f'x_tr_s shape: {x_tr_s.shape}')
print(f'x_te_s shape: {x_te_s.shape}')

Running feature selection...
x_tr_s shape: (326222, 1000)
x_te_s shape: (102111, 1000)


In [4]:
# Train models on a small sample
x_sample = x_tr_s[:5000]
y_sample = y_tr[:5000]

print(f'x_sample shape: {x_sample.shape}')
print(f'y_sample length: {len(y_sample)}')

x_sample shape: (5000, 1000)
y_sample length: 5000


## Grid Search Definitions

In [5]:
lr_grid = GridSearchCV(
    LogisticRegression(max_iter=2000, class_weight='balanced', random_state=42),
    {
        'C': [0.1, 1, 10],
        'solver': ['liblinear', 'lbfgs'],
    },
    cv = 5,
    scoring = 'f1',
    n_jobs = -1,
)

print('Defined Logistic Regression grid.')

Defined Logistic Regression grid.


In [6]:
svm_poly_grid = GridSearchCV(
    SVC(kernel='poly', probability=True, class_weight='balanced', random_state=42),
    {
        'C': [0.1, 1, 10],
        'degree': [2, 3],
        'gamma': ['scale', 'auto'],
    },
    cv=5,
    scoring='f1',
    n_jobs=-1,
)

print('Defined SVM Poly grid.')

Defined SVM Poly grid.


In [7]:
svm_rbf_grid = GridSearchCV(
    SVC(kernel='rbf', probability=True, class_weight='balanced', random_state=42),
    {
        'C': [0.1, 1, 10],
        'gamma': ['scale', 'auto'],
    },
    cv=5,
    scoring='f1',
    n_jobs=-1,
)

print('Defined SVM RBF grid.')

Defined SVM RBF grid.


In [8]:
knn_grid = GridSearchCV(
    KNeighborsClassifier(),
    {
        'n_neighbors': [3, 5, 7],
        'weights': ['uniform', 'distance'],
        'metric': ['minkowski', 'manhattan'],
    },
    cv=5,
    scoring='f1',
    n_jobs=-1,
)

print('Defined KNN grid.')

Defined KNN grid.


In [9]:
rf_grid = GridSearchCV(
    RandomForestClassifier(random_state=42),
    {
        'n_estimators': [100, 200],
        'max_depth': [None, 20],
        'min_samples_split': [2, 5],
    },
    cv=5,
    scoring='f1',
    n_jobs=-1,
)

print('Defined Random Forest grid.')

Defined Random Forest grid.


In [10]:
ridge_calibrated = CalibratedClassifierCV(
    RidgeClassifier(class_weight='balanced', random_state=42),
    method='sigmoid',
    cv=5,
)

ridge_grid = GridSearchCV(
    ridge_calibrated,
    {
        'estimator__alpha': [0.1, 1.0, 10.0],
    },
    cv=5,
    scoring='f1',
    n_jobs=-1,
)

print('Defined Ridge Classifier grid.')

Defined Ridge Classifier grid.


In [11]:
grid_jobs = {
    "Logistic_Regression": lr_grid,
    "SVM_Poly": svm_poly_grid,
    "SVM RBF": svm_rbf_grid,
    "KNN": knn_grid,
    "Random_Forest": rf_grid,
    "Ridge": ridge_grid
}

print('Tuning models...')
for model_name, grid in tqdm(grid_jobs.items(), total=len(grid_jobs), desc='Grid search'):
    grid.fit(x_sample, y_sample)
    print(f'  Tuned: {model_name}')
print('All grid searches complete.')

Tuning models...


Grid search:  17%|███████████▊                                                           | 1/6 [00:10<00:52, 10.59s/it]

  Tuned: Logistic_Regression


C:\Users\shrij\OneDrive\Desktop\spam_guard_1\.venv\Lib\site-packages\sklearnex\svm\_base.py:362: RuntimeWarning: random_state does not influence oneDAL SVM results
  warnings.warn(
Grid search:  33%|███████████████████████▋                                               | 2/6 [00:32<01:09, 17.50s/it]

  Tuned: SVM_Poly


...

  Tuned: KNN


Grid search:  83%|███████████████████████████████████████████████████████████▏           | 5/6 [01:26<00:19, 19.59s/it]

  Tuned: Random_Forest


Grid search: 100%|███████████████████████████████████████████████████████████████████████| 6/6 [01:27<00:00, 14.54s/it]

  Tuned: Ridge
All grid searches complete.


In [15]:
print('LR best params:', lr_grid.best_params_)
print('SVM Poly best params:', svm_poly_grid.best_params_)
print('SVM RBF best params:', svm_rbf_grid.best_params_)
print('KNN best params:', knn_grid.best_params_)
print('RF best params:', rf_grid.best_params_)
print('Ridge best params:', ridge_grid.best_params_)

LR best params: {'C': 10, 'solver': 'liblinear'}
SVM Poly best params: {'C': 10, 'degree': 2, 'gamma': 'scale'}
SVM RBF best params: {'C': 1, 'gamma': 'scale'}
KNN best params: {'metric': 'minkowski', 'n_neighbors': 3, 'weights': 'distance'}
RF best params: {'max_depth': None, 'min_samples_split': 5, 'n_estimators': 200}
Ridge best params: {'estimator__alpha': 1.0}


In [21]:
best_models = {
    'Logistic Regression': lr_grid.best_estimator_,
    'SVM Poly': svm_poly_grid.best_estimator_,
    'SVM RBF': svm_rbf_grid.best_estimator_,
    'KNN': knn_grid.best_estimator_,
    'Random Forest': rf_grid.best_estimator_,
    'Ridge Classifier': ridge_grid.best_estimator_,
}

for i in ['KNN', 'Random Forest', 'Ridge Classifier']:
    best_models[i] = best_models[i].set_params(n_jobs=-1)

best_models['Random Forest'] = best_models['Random Forest'].set_params(verbose=2)

print('Best model objects assembled.')

Best model objects assembled.


In [ ]:
print('Training tuned models on full training set...')
for model_name, model in tqdm(best_models.items(), total=len(best_models), desc='Training base models'):
    model.fit(x_tr_s, y_tr)
    print(f'  Trained: {model_name}')
    joblib.dump(model, f'../models/model_{model_name}.pkl')

Training tuned models on full training set...


Training base models:  17%|██████████▎                                                   | 1/6 [00:07<00:37,  7.54s/it]

  Trained: Logistic Regression


C:\Users\shrij\OneDrive\Desktop\spam_guard_1\.venv\Lib\site-packages\sklearnex\svm\_base.py:362: RuntimeWarning: random_state does not influence oneDAL SVM results
  warnings.warn(
Training base models:  33%|████████████████████▎                                        | 2/6 [11:44<27:33, 413.26s/it]

  Trained: SVM Poly


...

building tree 151 of 200
building tree 152 of 200
building tree 153 of 200
building tree 154 of 200
building tree 155 of 200
building tree 156 of 200
building tree 157 of 200
building tree 158 of 200
building tree 159 of 200
building tree 160 of 200
building tree 161 of 200
building tree 162 of 200
building tree 163 of 200
building tree 164 of 200
building tree 165 of 200
building tree 166 of 200
building tree 167 of 200
building tree 168 of 200
building tree 169 of 200building tree 170 of 200

building tree 171 of 200
building tree 172 of 200
building tree 173 of 200
building tree 174 of 200
building tree 175 of 200
building tree 176 of 200
building tree 177 of 200
building tree 178 of 200
building tree 179 of 200
building tree 180 of 200
building tree 181 of 200
building tree 182 of 200
building tree 183 of 200
building tree 184 of 200
building tree 185 of 200
building tree 186 of 200
building tree 187 of 200
building tree 188 of 200
building tree 189 of 200
building tree 190 of 200


[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed: 13.6min finished


  Trained: Random Forest


Training base models: 100%|█████████████████████████████████████████████████████████████| 6/6 [32:22<00:00, 323.73s/it]

  Trained: Ridge Classifier


In [23]:
print('Building ensemble...')
ensemble = VotingClassifier(
    estimators=[
        ('lr', best_models['Logistic Regression']),
        ('svm_poly', best_models['SVM Poly']),
        ('svm_rbf', best_models['SVM RBF']),
        ('knn', best_models['KNN']),
        ('rf', best_models['Random Forest']),
        ('ridge', best_models['Ridge Classifier']),
    ],
    voting='soft',
)
ensemble.fit(x_tr_s, y_tr)
print('Ensemble training complete.')

Building ensemble...


C:\Users\shrij\OneDrive\Desktop\spam_guard_1\.venv\Lib\site-packages\sklearnex\svm\_base.py:362: RuntimeWarning: random_state does not influence oneDAL SVM results
  warnings.warn(
C:\Users\shrij\OneDrive\Desktop\spam_guard_1\.venv\Lib\site-packages\sklearnex\svm\_base.py:362: RuntimeWarning: random_state does not influence oneDAL SVM results
  warnings.warn(
[Parallel(n_jobs=-1)]: Using backend ThreadingBackend with 12 concurrent workers.


building tree 12 of 200
building tree 5 of 200
building tree 6 of 200
building tree 4 of 200
building tree 10 of 200
building tree 11 of 200
building tree 1 of 200
building tree 8 of 200
building tree 9 of 200
building tree 2 of 200
building tree 7 of 200
building tree 3 of 200
building tree 13 of 200
building tree 14 of 200
building tree 15 of 200
building tree 16 of 200
building tree 17 of 200
building tree 18 of 200
building tree 19 of 200
building tree 20 of 200
building tree 21 of 200building tree 22 of 200

building tree 23 of 200
building tree 24 of 200
building tree 25 of 200
building tree 26 of 200
building tree 27 of 200
building tree 28 of 200
building tree 29 of 200
building tree 30 of 200


[Parallel(n_jobs=-1)]: Done  17 tasks      | elapsed:  1.6min


building tree 31 of 200
building tree 32 of 200
building tree 33 of 200
building tree 34 of 200
building tree 35 of 200
building tree 36 of 200
building tree 37 of 200
building tree 38 of 200
building tree 39 of 200
building tree 40 of 200
building tree 41 of 200
building tree 42 of 200
building tree 43 of 200
building tree 44 of 200
building tree 45 of 200
building tree 46 of 200
building tree 47 of 200
building tree 48 of 200
building tree 49 of 200
building tree 50 of 200
building tree 51 of 200
building tree 52 of 200
building tree 53 of 200
building tree 54 of 200
building tree 55 of 200
building tree 56 of 200
building tree 57 of 200
building tree 58 of 200
building tree 59 of 200
building tree 60 of 200
building tree 61 of 200
building tree 62 of 200
building tree 63 of 200
building tree 64 of 200
building tree 65 of 200
building tree 66 of 200
building tree 67 of 200
building tree 68 of 200
building tree 69 of 200
building tree 70 of 200
building tree 71 of 200
building tree 72

[Parallel(n_jobs=-1)]: Done 138 tasks      | elapsed:  9.7min


building tree 151 of 200
building tree 152 of 200
building tree 153 of 200
building tree 154 of 200
building tree 155 of 200
building tree 156 of 200
building tree 157 of 200
building tree 158 of 200
building tree 159 of 200
building tree 160 of 200
building tree 161 of 200
building tree 162 of 200
building tree 163 of 200
building tree 164 of 200
building tree 165 of 200
building tree 166 of 200
building tree 167 of 200
building tree 168 of 200
building tree 169 of 200
building tree 170 of 200
building tree 171 of 200
building tree 172 of 200
building tree 173 of 200
building tree 174 of 200
building tree 175 of 200
building tree 176 of 200
building tree 177 of 200
building tree 178 of 200
building tree 179 of 200
building tree 180 of 200
building tree 181 of 200
building tree 182 of 200
building tree 183 of 200
building tree 184 of 200
building tree 185 of 200
building tree 186 of 200
building tree 187 of 200
building tree 188 of 200
building tree 189 of 200
building tree 190 of 200


[Parallel(n_jobs=-1)]: Done 200 out of 200 | elapsed: 13.6min finished


Ensemble training complete.


In [24]:
joblib.dump(ensemble, f'../models/ensemble_model.pkl')
joblib.dump(sel, f'../models/sel.pkl')
print(f"Success! Model and Selector saved")

Success! Model and Selector saved


In [2]:
!jupyter nbconvert --to script model_training.ipynb

[NbConvertApp] Converting notebook model_training.ipynb to script
[NbConvertApp] Writing 5828 bytes to model_training.py
